In [ ]:
import os
from io import BytesIO
from astropy.io import fits
import matplotlib.pyplot as plt
import dropbox
import time
import glob
import warnings
import numpy as np
import pandas as pd

from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
from astropy.coordinates import SkyCoord
from astropy.nddata import Cutout2D
from astropy.table import Table
from astropy.wcs import FITSFixedWarning
from scipy import ndimage

In [ ]:
warnings.simplefilter('ignore', FITSFixedWarning)

In [ ]:
# Links to shared folders
dbx_url = 'https://www.dropbox.com/scl/fo/37ooho4m924wb3d2m1gt8/ABJjd8gNUl0h_rmUP41S3cI?rlkey=nqy0t7p9sgxa3a853bf1ris9l&st=clxv8yui&dl=0'
# jwst_url = 'https://www.dropbox.com/scl/fo/eo3ktfmhniyqm7l32egp4/ANfH2VisN0vIphVxNIpO_ag/JWST?rlkey=enfshikpl0r0wd8o2nz66i0km&subfolder_nav_tracking=1&st=w3u4yz80&dl=0'
# nisp_url = 'https://www.dropbox.com/scl/fo/eo3ktfmhniyqm7l32egp4/AEiIbqfbMKqh6FRA2YouLHQ/NISP-J?rlkey=enfshikpl0r0wd8o2nz66i0km&subfolder_nav_tracking=1&st=hx3ru322&dl=0'

In [ ]:
def get_shared_folder_metadata(url, dbx, path=""):
    """Get metadata of the shared folder."""
    shared_link = dropbox.files.SharedLink(url=url)
    try:
        folder_metadata = dbx.files_list_folder(path=f'{path}', shared_link=shared_link)
        return folder_metadata.entries
    except dropbox.exceptions.ApiError as e:
        print(f"Error accessing shared folder: {e}")
        return []

def get_fits_file(url, file_name, dbx, hdu_idx=0):
    tries=5
    while tries >= 0:
        tries -= 1
        try:
            meta, res = dbx.sharing_get_shared_link_file(url, path=file_name)
            break
        except Exception as e:
            print(f"Connection error: {e}")
            time.sleep(10)
            continue

    try: res
    except: raise RuntimeError(f"Could not access file {file_name} at URL {url}.")

    fits_file = BytesIO(res.content)
    hdul = fits.open(fits_file)

    # Access image data
    image_data = hdul[hdu_idx].data
    image_header = hdul[hdu_idx].header
    hdul.close()
    return image_header, image_data

def cut_catalog(cat_file, cuts=None):
    # Open cat file
    with fits.open(cat_file) as hdul:
        cat_data = hdul[1].data  # Adjust the HDU index if your data is not in the first extension
    
    # Do cuts on catalog data
    if cuts is None:
        cuts = (cat_data.lp_type==0) & \
                (cat_data.ACS_F814W_MAG < 25) & \
                (cat_data.ez_z_phot > 0.01) & (cat_data.ez_z_phot < 3.0) & \
                (cat_data.FLUX_RADIUS < 104) # TODO: make this a function with J&H bands?
    
    cat_clipped = cat_data[cuts]
    
    my_cat = pd.DataFrame({'id_classic': cat_clipped['ID'].astype(int),
                           'ra': cat_clipped['ALPHA_J2000'].astype(float),
                           'dec': cat_clipped['DELTA_J2000'].astype(float),
                           'jwst_image': "",
                           'nisp_image': "",
                          })
    
    return my_cat

def cut_catalog2(cat_file, cuts=None):

    cat_data = Table.read(cat_file).to_pandas()
    
    cat_data['ACS_F814W_MAG'] = 23.9-2.5*np.log10(cat_data.ACS_F814W_FLUX)
    cat_clipped = cat_data[
            (cat_data.z_best > 0.01) & (cat_data.z_best < 3.0)
            & (cat_data.ACS_F814W_MAG < 25)
            & (cat_data.FLUX_RADIUS_2_F814W < 50) & (cat_data.FLUX_RADIUS_2_F814W > 3) # Can be flexible with this (remove or make much larger)
            & (cat_data.CLASS_STAR < 0.1)
    ]

    my_cat = pd.DataFrame({
        'id': cat_clipped['ID'],
        'ra': cat_clipped['RA_1'],
        'dec': cat_clipped['DEC_1'],
        'jwst_image': '',
        'nisp_image': '',
    })

    my_cat = my_cat.dropna().reset_index(drop=True)
    return my_cat

def match_catalog(file_name, gal_coords, hdu_idx=0, url=None, dbx=None):
    if url is None or dbx is None: # get images locally
        if not os.path.exists(file_name):
            raise ValueError(f"File name '{file_name}' does not exist. " \
                                "Provide URL and dbx instance for dropbox usage.")
        with fits.open(file_name) as hdul:
            image_header = hdul[hdu].header
            image_data = hdul[hdu].data
    else: # Get FITS file from dropbox
        image_header, image_data = get_fits_file(url, file_name, dbx=dbx, 
                                                 hdu_idx=hdu_idx)
    
    wcs = WCS(image_header)
    
    return np.where(gal_coords.contained_by(wcs))[0]

def rotate_jwst(clip, angle=-20, size=66, pad=20):
    # There are different types of interpolation possible for this one, talk about it with Shooby
    wcs = clip.wcs
    new_image = ndimage.rotate(clip.data, angle, reshape=False, cval=-20)
    new_clip = Cutout2D(new_image, clip.center_cutout, size)
    # new_clip.wcs = wcs
    return new_clip # no WCS info after this

# def rotate_wcs(clip, angle=-20):
#     # Define rotation angle (in degrees)
#     theta = np.radians(angle)
#     wcs = clip.wcs.deepcopy()
    
#     # Create a 2D rotation matrix
#     rotation_matrix = np.array([[np.cos(theta), -np.sin(theta)],
#                                 [np.sin(theta),  np.cos(theta)]])
    
#     # Get the center of the WCS frame
#     coord_center = wcs.pixel_to_world(clip.center_cutout[1], clip.center_cutout[0])
#     x_center, y_center = coord_center.ra.degree, coord_center.dec.degree
    
#     # Translation matrices
#     translation_to_origin = np.array([[-x_center], [-y_center]])
#     translation_back = np.array([[x_center], [y_center]])
    
#     # Apply translation, rotation, and reverse translation
#     if hasattr(wcs.wcs, 'pc'):
#         wcs.wcs.pc = translation_back + np.dot(rotation_matrix, (wcs.wcs.pc + translation_to_origin))
#     elif hasattr(wcs.wcs, 'cd'):
#         wcs.wcs.cd = translation_back + np.dot(rotation_matrix, (wcs.wcs.cd + translation_to_origin))

#     return wcs

def clip_images(catalog, url=None, dbx=None, size_jwst=66, size_nisp=40, pad=20, 
                rot_jwst=-20.0, limit=None, jwst_hdu=0, normalize=1):
    ### The catalog must have matched ra, dec, jwst_image, and nisp_image columns
    image_pairs = catalog.groupby(['jwst_image', 'nisp_image'])

    clips = []
    count = 0
    for pair, cat in image_pairs:
        jwst_file, nisp_file = pair

        # Open JWST and Euclid images
        if dbx is None or url is None: # Open from file
            
            # Check files exist
            if not os.path.exists(jwst_file):
                raise ValueError(f"File name '{jwst_file}' does not exist. " \
                                "Provide URL and dbx instance for dropbox usage.")
            if not os.path.exists(nisp_file):
                raise ValueError(f"File name '{nisp_file}' does not exist. " \
                                "Provide URL and dbx instance for dropbox usage.")

            # Open JWST and NISP files from disk
            with fits.open(jwst_file) as hdul:
                jwst_header = hdul[jwst_hdu].header
                jwst_data = hdul[jwst_hdu].data
            with fits.open(nisp_file) as hdul:
                nisp_header = hdul[0].header
                nisp_data = hdul[0].data
                
        else: # Open from dropbox
            print('Fetching files from dropbox...')
            jwst_header, jwst_data = get_fits_file(url, jwst_file, dbx=dbx, hdu_idx=jwst_hdu)
            nisp_header, nisp_data = get_fits_file(url, nisp_file, dbx=dbx)
        
        wcs_jwst, wcs_nisp = WCS(jwst_header), WCS(nisp_header)
        
        # Get the coordinates of matched galaxies
        gal_coords = SkyCoord(cat.ra, cat.dec, unit='deg')
        
        for i in range(len(cat)):
            gal = cat.iloc[i]
            # Get JWST clip (larger than final size)
            clip_jwst = Cutout2D(jwst_data, gal_coords[i], size=size_jwst+pad, wcs=wcs_jwst, mode='trim')
            if sum(clip_jwst.data.shape)!=(size_jwst+pad)*2: continue
            if np.sum((clip_jwst.data==0.0).astype(int))/((size_jwst+pad)**2) > 0.25: continue
            
            # Get NISP clip
            clip_nisp = Cutout2D(nisp_data, gal_coords[i], size=size_nisp, wcs=wcs_nisp, mode='trim')
            if sum(clip_nisp.data.shape)!=(size_nisp)*2: continue
            if np.sum((clip_nisp.data==0.0).astype(int))/((size_nisp)**2) > 0.25: continue
            # clip_nisp.data = ABmag_nisp(clip_nisp, nisp_header['MAGZERO'])
            
            # Rotate JWST image 20 degrees counter-clockwise and crop
            # This loses the WCS
            if rot_jwst != 0:
                clip_jwst = rotate_jwst(clip_jwst, size=size_jwst, angle=rot_jwst)
                # clip_jwst.wcs = rotate_wcs(clip_jwst, angle=rot_jwst)
            # clip_jwst.data = ABmag_jwst(clip_jwst, jwst_header['PIXAR_SR'])

            if normalize:
                da_lr = np.arcsinh(clip_nisp)
                clip_nisp = (da_lr - da_lr.min()) / (da_lr.max() - da_lr.min() + 1e-8)

                da_hr = np.arcsinh(clip_jwst)
                clip_jwst = (da_hr - da_hr.min()) / (da_hr.max() - da_hr.min() + 1e-8)
                
            
            clips.append((gal_coords[i], clip_jwst, clip_nisp))
            count += 1
            if limit is not None and count >= limit: break
        if limit is not None and count >= limit: break

    return clips

In [ ]:
meta = {
    'COSMOS': {
        'cut_func': cut_catalog,
        'size_jwst': 66,
        'size_nisp': 40,
        'pad_jwst': 20,
        'rot_jwst': -20,
        'jwst_hdu': 0,
        'cat_path': '../catalog/COSMOS2020_CLASSIC_R1_v2.2_p3.fits', # COSMOS classic
        'matched_cat': '../catalog/matched_cat_COSMOS.csv',
        'matched_jwst': '../data/jwst_cosmos_66px.npy',
        'matched_nisp': '../data/nisp_cosmos_40px.npy',
    },
    'HUDF': {
        'cut_func': cut_catalog2,
        'size_jwst': 134,
        'size_nisp': 40,
        'pad_jwst': 0,
        'rot_jwst': 0,
        'jwst_hdu': 1,
        'cat_path': '../catalog/gds.fits', # CANDELS catalog
        'matched_cat': '../catalog/matched_cat_HUDF.csv',
        'matched_jwst': '../data/jwst_hudf_66px.npy',
        'matched_nisp': '../data/jwst_hudf_66px.npy',
    }
}

In [ ]:
def process_all(field='COSMOS', save_cat=False, save_clips=False, secret="../../secrets/dropbox_token"):
    params = meta[field]
    # Instantiate dropbox token
    with open(secret) as token_file:
        token = token_file.read()
        dbx = dropbox.Dropbox(token.strip(), timeout=None)

    # Get JWST files for specific field
    jwst_path = f'/JWST/{field}/'
    jwst_files = get_shared_folder_metadata(dbx_url, path=jwst_path, dbx=dbx)
    jwst_files = [jwst_path+file.name for file in jwst_files]

    # Get NISP files for specific field
    nisp_path = f'/NISP-Y/{field}/'
    nisp_files = get_shared_folder_metadata(dbx_url, path=nisp_path, dbx=dbx)
    nisp_files = [nisp_path+file.name for file in nisp_files]
    
    # Apply cuts to the catalog
    my_cat = params['cut_func'](params['cat_path'])
    gal_coords = SkyCoord(my_cat.ra, my_cat.dec, unit='deg')

    # Match JWST files
    for file in jwst_files:
        found_idxs = match_catalog(file, gal_coords, url=dbx_url, dbx=dbx, 
                                   hdu_idx=params['jwst_hdu'])
        my_cat.loc[found_idxs, 'jwst_image'] = str(file)

    # Match NISP files
    for file in nisp_files:
        found_idxs = match_catalog(file, gal_coords, url=dbx_url, dbx=dbx)
        my_cat.loc[found_idxs, 'nisp_image'] = str(file)

    my_cat = my_cat[my_cat.nisp_image!='']
    my_cat = my_cat[my_cat.jwst_image!='']

    if save_cat:
        my_cat.to_csv(params['matched_cat'], index=False)

    try: my_cat
    except: my_cat = pd.read_csv(params['matched_cat'])
    clips = clip_images(my_cat, url=dbx_url, dbx=dbx, jwst_hdu=params['jwst_hdu'], 
                        size_jwst=params['size_jwst'], rot_jwst=params['rot_jwst'], 
                        pad=params['pad_jwst'])
    
    # Arrange data
    jwst_cutouts = np.array([clip[1].data for clip in clips])
    nisp_cutouts = np.array([clip[2].data for clip in clips])
    
    if save_clips:
        np.save(params['matched_jwst'], jwst_cutouts, allow_pickle=False)
        np.save(params['matched_nisp'], nisp_cutouts, allow_pickle=False)
    
    return clips

In [ ]:
clips = process_all('HUDF', save_cat=True, save_clips=True)

In [ ]:
jwst_cutouts = np.array([clip[1].data for clip in clips])
nisp_cutouts = np.array([clip[2].data for clip in clips])

In [ ]:
# Plotting images for verification purposes
n_rows = 6
fig, axes = plt.subplots(n_rows, 4, figsize=(10,2.5*n_rows))
for i in range(n_rows*2):
    clip_jwst = jwst_cutouts[i]
    clip_nisp = nisp_cutouts[i]
    axes.flatten()[i*2].imshow(clip_jwst)
    axes.flatten()[i*2+1].imshow(clip_nisp)
# plt.savefig("../plots/euclid_nisp_matches.png", bbox_inches='tight')